## tree-sitter: Preliminary Investigation

This notebook focus on experimentation to conduct preliminary investigation to deepen knownledge and understanding on `tree-sitter`.

Author: Boon Heng Chew <<s2831745@sms.ed.ac.uk>>

In [1]:
from platform import python_version

# show environment
print(f"Python environment: {python_version()}")

Python environment: 3.12.11


In [2]:
# Setup tree sitter dependencies
import tree_sitter_python as tspython
import tree_sitter_java as tsjava
from tree_sitter import Language, Parser, Query, QueryCursor

In [3]:
# Common subroutine to parse code
def print_cst(node, indent=0):
    """Recursively print the tree structure"""
    print("  " * indent + f"{node.type} [{node.start_point}-{node.end_point}]")
    for child in node.children:
        print_cst(child, indent + 1)

def parse_code(parser, code):
    tree = parser.parse(code)
    root = tree.root_node
    
    print("Root node type:", root.type)
    print("Root node:", root)
    print("\nCST:")
    print_cst(root)

### Exploration with Python

This section contain the experiment with tree-sitter on Python Language. Compare to ast, it is much more `low` level and contains elements beyond typical AST. Work needed to transform it, and to solely pull out the semantic elements

In [4]:
# Exploration with Python

# Load the Python language
PY_LANGUAGE = Language(tspython.language())

# Create a parser and set the language
python_parser = Parser(PY_LANGUAGE)

python_code = b'''
def main():
    """Prints hello world"""
    print("Hello World!")

if __name__ == "__main__":
    main()
'''

parse_code(python_parser, python_code)

Root node type: module
Root node: (module (function_definition name: (identifier) parameters: (parameters) body: (block (expression_statement (string (string_start) (string_content) (string_end))) (expression_statement (call function: (identifier) arguments: (argument_list (string (string_start) (string_content) (string_end))))))) (if_statement condition: (comparison_operator (identifier) (string (string_start) (string_content) (string_end))) consequence: (block (expression_statement (call function: (identifier) arguments: (argument_list))))))

CST:
module [Point(row=1, column=0)-Point(row=7, column=0)]
  function_definition [Point(row=1, column=0)-Point(row=3, column=25)]
    def [Point(row=1, column=0)-Point(row=1, column=3)]
    identifier [Point(row=1, column=4)-Point(row=1, column=8)]
    parameters [Point(row=1, column=8)-Point(row=1, column=10)]
      ( [Point(row=1, column=8)-Point(row=1, column=9)]
      ) [Point(row=1, column=9)-Point(row=1, column=10)]
    : [Point(row=1, co

In [5]:
# Create the tree-sitter query to pull out semantic elements
pyquery = Query(PY_LANGUAGE, """
(function_definition
  name: (identifier) @func.name
  parameters: (parameters) @func.params
  body: (block) @func.body)
""")

python_tree = python_parser.parse(python_code)
python_root = python_tree.root_node

# Call captures on the node, passing the query
cursor = QueryCursor(pyquery)
captures = cursor.captures(python_root)

print("Captured nodes:")
for capture_name, nodes in captures.items():
    print(f"\n{capture_name}:")
    for node in nodes:
        print(f"  Type: {node.type}")
        print(f"  Text: {node.text.decode('utf-8')[:50]}")
        print(f"  Position: {node.start_point} -> {node.end_point}")

Captured nodes:

func.name:
  Type: identifier
  Text: main
  Position: Point(row=1, column=4) -> Point(row=1, column=8)

func.params:
  Type: parameters
  Text: ()
  Position: Point(row=1, column=8) -> Point(row=1, column=10)

func.body:
  Type: block
  Text: """Prints hello world"""
    print("Hello World!")
  Position: Point(row=2, column=4) -> Point(row=3, column=25)


In [6]:
import ast

ast_tree = ast.parse(python_code)
print(ast.dump(ast_tree, indent=2))

def print_ast(node, indent=0):
    """Recursively print AST structure"""
    node_name = node.__class__.__name__
    print("  " * indent + node_name, end="")
    
    # Print some useful attributes
    if hasattr(node, 'name'):
        print(f" (name={node.name})", end="")
    if hasattr(node, 'id'):
        print(f" (id={node.id})", end="")
    if isinstance(node, ast.Constant):
        print(f" (value={repr(node.value)})", end="")
    print()
    
    for child in ast.iter_child_nodes(node):
        print_ast(child, indent + 1)

print_ast(ast_tree)

Module(
  body=[
    FunctionDef(
      name='main',
      args=arguments(
        posonlyargs=[],
        args=[],
        kwonlyargs=[],
        kw_defaults=[],
        defaults=[]),
      body=[
        Expr(
          value=Constant(value='Prints hello world')),
        Expr(
          value=Call(
            func=Name(id='print', ctx=Load()),
            args=[
              Constant(value='Hello World!')],
            keywords=[]))],
      decorator_list=[],
      type_params=[]),
    If(
      test=Compare(
        left=Name(id='__name__', ctx=Load()),
        ops=[
          Eq()],
        comparators=[
          Constant(value='__main__')]),
      body=[
        Expr(
          value=Call(
            func=Name(id='main', ctx=Load()),
            args=[],
            keywords=[]))],
      orelse=[])],
  type_ignores=[])
Module
  FunctionDef (name=main)
    arguments
    Expr
      Constant (value='Prints hello world')
    Expr
      Call
        Name (id=print)
          Load


In [7]:
from dataclasses import dataclass
from typing import List, Any

# Define AST-like structures
@dataclass
class Module:
    body: List[Any]

@dataclass
class FunctionDef:
    name: str
    params: List[str]
    body: List[Any]

@dataclass
class Call:
    func: str
    args: List[Any]

@dataclass
class If:
    test: Any
    body: List[Any]

def convert_to_ast(node):
    """Convert tree-sitter node to AST-like structure"""
    
    if node.type == 'module':
        body = []
        for child in node.children:
            if child.is_named:  # Skip punctuation
                ast_child = convert_to_ast(child)
                if ast_child:
                    body.append(ast_child)
        return Module(body=body)
    
    elif node.type == 'function_definition':
        name = node.child_by_field_name('name').text.decode('utf-8')
        params_node = node.child_by_field_name('parameters')
        params = [p.text.decode('utf-8') for p in params_node.named_children]
        
        body_node = node.child_by_field_name('body')
        body = [convert_to_ast(stmt) for stmt in body_node.named_children]
        
        return FunctionDef(name=name, params=params, body=body)
    
    elif node.type == 'expression_statement':
        # Handle calls inside expressions
        for child in node.children:
            if child.type == 'call':
                return convert_to_ast(child)
            elif child.type == 'string':
                return child.text.decode('utf-8')  # Docstring
    
    elif node.type == 'call':
        func_node = node.child_by_field_name('function')
        func = func_node.text.decode('utf-8')
        
        args_node = node.child_by_field_name('arguments')
        args = [convert_to_ast(arg) for arg in args_node.named_children]
        
        return Call(func=func, args=args)
    
    elif node.type == 'if_statement':
        condition = node.child_by_field_name('condition')
        body_node = node.child_by_field_name('consequence')
        body = [convert_to_ast(stmt) for stmt in body_node.named_children]
        
        return If(test=condition.text.decode('utf-8'), body=body)
    
    elif node.type == 'string':
        return node.text.decode('utf-8')
    
    return None

# Convert and print
ast_tree = convert_to_ast(python_root)

def pretty_print_ast(node, indent=0):
    """Pretty print AST-like structure similar to ast.dump()"""
    spaces = "  " * indent
    
    if isinstance(node, Module):
        print(f"{spaces}Module(")
        print(f"{spaces}  body=[")
        for item in node.body:
            pretty_print_ast(item, indent + 2)
        print(f"{spaces}  ])")
    
    elif isinstance(node, FunctionDef):
        print(f"{spaces}FunctionDef(")
        print(f"{spaces}  name='{node.name}',")
        print(f"{spaces}  params={node.params},")
        print(f"{spaces}  body=[")
        for item in node.body:
            pretty_print_ast(item, indent + 2)
        print(f"{spaces}  ])")
    
    elif isinstance(node, Call):
        print(f"{spaces}Call(")
        print(f"{spaces}  func='{node.func}',")
        print(f"{spaces}  args=[")
        for arg in node.args:
            if isinstance(arg, str):
                print(f"{spaces}    '{arg}',")
            else:
                pretty_print_ast(arg, indent + 2)
        print(f"{spaces}  ])")
    
    elif isinstance(node, If):
        print(f"{spaces}If(")
        print(f"{spaces}  test='{node.test}',")
        print(f"{spaces}  body=[")
        for item in node.body:
            pretty_print_ast(item, indent + 2)
        print(f"{spaces}  ])")
    
    elif isinstance(node, str):
        print(f"{spaces}Constant(value='{node}')")
    
    else:
        print(f"{spaces}{node}")

# Use it
pretty_print_ast(ast_tree)

Module(
  body=[
    FunctionDef(
      name='main',
      params=[],
      body=[
        Constant(value='"""Prints hello world"""')
        Call(
          func='print',
          args=[
            '"Hello World!"',
          ])
      ])
    If(
      test='__name__ == "__main__"',
      body=[
        Call(
          func='main',
          args=[
          ])
      ])
  ])


### Exploration with Java

This section contains experiments with tree-sitter on Java. It covers generics, lambda expressions, and Java 17+ records — including the `record` syntax that `javalang` cannot parse.

In [8]:
# Exploration with Java

# Load the JAva language
JAVA_LANGUAGE = Language(tsjava.language())

# Create a parser and set the language
java_parser = Parser(JAVA_LANGUAGE)

java_code = b'''
import java.util.List;
import java.util.ArrayList;
import java.util.function.Predicate;

public class GenericLambdaExample<T> {
    private List<T> items;
    
    public GenericLambdaExample() {
        this.items = new ArrayList<>();
    }
    
    public void addItem(T item) {
        items.add(item);
    }
    
    public List<T> filter(Predicate<T> predicate) {
        List<T> result = new ArrayList<>();
        for (T item : items) {
            if (predicate.test(item)) {
                result.add(item);
            }
        }
        return result;
    }
    
    public static void main(String[] args) {
        GenericLambdaExample<String> example = new GenericLambdaExample<>();
        example.addItem("hello");
        example.addItem("world");
        example.addItem("foo");
        
        // Lambda expression
        List<String> filtered = example.filter(s -> s.length() > 3);
        filtered.forEach(System.out::println);
    }
}
'''

parse_code(java_parser, java_code)

Root node type: program
Root node: (program (import_declaration (scoped_identifier scope: (scoped_identifier scope: (identifier) name: (identifier)) name: (identifier))) (import_declaration (scoped_identifier scope: (scoped_identifier scope: (identifier) name: (identifier)) name: (identifier))) (import_declaration (scoped_identifier scope: (scoped_identifier scope: (scoped_identifier scope: (identifier) name: (identifier)) name: (identifier)) name: (identifier))) (class_declaration (modifiers) name: (identifier) type_parameters: (type_parameters (type_parameter (type_identifier))) body: (class_body (field_declaration (modifiers) type: (generic_type (type_identifier) (type_arguments (type_identifier))) declarator: (variable_declarator name: (identifier))) (constructor_declaration (modifiers) name: (identifier) parameters: (formal_parameters) body: (constructor_body (expression_statement (assignment_expression left: (field_access object: (this) field: (identifier)) right: (object_creatio

In [9]:
# Create the query
query = Query(JAVA_LANGUAGE, """
(method_declaration
  name: (identifier) @method.name
  parameters: (formal_parameters) @method.params
  body: (block) @method.body)
""")

java_tree = java_parser.parse(java_code)
java_root = java_tree.root_node

# Call captures on the node, passing the query
cursor = QueryCursor(query)
captures = cursor.captures(java_root)

print("Captured nodes:")
for capture_name, nodes in captures.items():
    print(f"\n{capture_name}:")
    for node in nodes:
        print(f"  Type: {node.type}")
        print(f"  Text: {node.text.decode('utf-8')[:50]}")
        print(f"  Position: {node.start_point} -> {node.end_point}")

Captured nodes:

method.name:
  Type: identifier
  Text: addItem
  Position: Point(row=12, column=16) -> Point(row=12, column=23)
  Type: identifier
  Text: filter
  Position: Point(row=16, column=19) -> Point(row=16, column=25)
  Type: identifier
  Text: main
  Position: Point(row=26, column=23) -> Point(row=26, column=27)

method.params:
  Type: formal_parameters
  Text: (T item)
  Position: Point(row=12, column=23) -> Point(row=12, column=31)
  Type: formal_parameters
  Text: (Predicate<T> predicate)
  Position: Point(row=16, column=25) -> Point(row=16, column=49)
  Type: formal_parameters
  Text: (String[] args)
  Position: Point(row=26, column=27) -> Point(row=26, column=42)

method.body:
  Type: block
  Text: {
        items.add(item);
    }
  Position: Point(row=12, column=32) -> Point(row=14, column=5)
  Type: block
  Text: {
        List<T> result = new ArrayList<>();
    
  Position: Point(row=16, column=50) -> Point(row=24, column=5)
  Type: block
  Text: {
        GenericLa

#### Java 17+ Record Support

The same `record` class that causes `javalang` to throw a `JavaSyntaxError` is parsed here to confirm tree-sitter handles modern Java syntax correctly.

In [10]:
# Test tree-sitter on a Java 17 record — the same construct that causes javalang to fail
java_record_code = b'''
public record Point(int x, int y) {
    public double distanceTo(Point other) {
        return Math.sqrt(Math.pow(this.x - other.x, 2) + Math.pow(this.y - other.y, 2));
    }
}
'''

print("=== CST for Java 17 record ===")
parse_code(java_parser, java_record_code)

# Extract methods using the same query as above
record_tree = java_parser.parse(java_record_code)
record_root = record_tree.root_node

cursor = QueryCursor(query)
captures = cursor.captures(record_root)

print("\n=== Extracted methods from record ===")
if captures:
    for capture_name, nodes in captures.items():
        print(f"\n{capture_name}:")
        for node in nodes:
            print(f"  Text: {node.text.decode('utf-8')[:80]}")
else:
    print("No methods captured.")

=== CST for Java 17 record ===
Root node type: program
Root node: (program (record_declaration (modifiers) name: (identifier) parameters: (formal_parameters (formal_parameter type: (integral_type) name: (identifier)) (formal_parameter type: (integral_type) name: (identifier))) body: (class_body (method_declaration (modifiers) type: (floating_point_type) name: (identifier) parameters: (formal_parameters (formal_parameter type: (type_identifier) name: (identifier))) body: (block (return_statement (method_invocation object: (identifier) name: (identifier) arguments: (argument_list (binary_expression left: (method_invocation object: (identifier) name: (identifier) arguments: (argument_list (binary_expression left: (field_access object: (this) field: (identifier)) right: (field_access object: (identifier) field: (identifier))) (decimal_integer_literal))) right: (method_invocation object: (identifier) name: (identifier) arguments: (argument_list (binary_expression left: (field_access object:

#### Finding: tree-sitter handles Java 17+ records

Unlike `javalang`, tree-sitter successfully parses the `record` declaration. The `record_declaration` node is correctly identified in the CST, and the query API extracts the record's method normally. This confirms tree-sitter as the viable path for handling modern Java syntax in future work.